## Simple neural network for sentiment anlysis on movie review

This notebook explores the different tools to build the most efficient neural network for sentiment anlysis on movie reviews. The actual running of the code will be done in simple python files to be able to run them using wandb and the GPU put at our disposition.

### Imports

In [1]:
import pandas as pd
import numpy as np 
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as torchdata
import matplotlib.pyplot as plt


### Data extraction

In [2]:
data = pd.read_csv("../rotten_tomatoes_movie_reviews.csv")
data = data.dropna(subset=["reviewText"])

sentiment_mapping = {'POSITIVE' : 0, 'NEGATIVE' : 1}
data["scoreSentiment"] = data["scoreSentiment"].map(sentiment_mapping).to_numpy()

In [4]:
reviews = data["reviewText"].to_numpy()
scores = data["originalScore"].to_numpy()
sentiments = data["scoreSentiment"]
print(reviews[:5])
print(sentiments[:5])

["Timed to be just long enough for most youngsters' brief attention spans -- and it's packed with plenty of interesting activity, both on land and under the water."
 "It doesn't matter if a movie costs 300 million or only 300 dollars; good is good and bad is bad, and Bloodmask: The Possession of Nicole Lameroux is just plain bad."
 "The choreography is so precise and lifelike at points one might wonder whether the movie was rotoscoped, but no live-action reference footage was used. The quality is due to the skill of the animators and Kodama's love for professional wrestling."
 "The film's out-of-touch attempts at humor may find them hunting for the reason the franchise was so popular in the first place."
 'Its clumsy determination is endearing and sometimes wildly entertaining']
0    0
1    1
2    0
3    1
4    0
Name: scoreSentiment, dtype: int64


### Data preprocessing

In [5]:
def clean_text(text):
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

data["reviewText"] = data["reviewText"].apply(clean_text)

In [ ]:
X = data["reviewText"].to_numpy()

y = data["scoreSentiment"].to_numpy()

print(type(X))

<class 'numpy.ndarray'>


In [13]:
print(X[0])
print(len(str(X[0]).split()))
max_word_count = max(len(str(review).split()) for review in X)
print("Maximum number of words in a review:", max_word_count)

timed to be just long enough for most youngsters brief attention spans and it s packed with plenty of interesting activity both on land and under the water
28
Maximum number of words in a review: 73


### Tokenization and padding

In [7]:
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X)
X_sequences = tokenizer.texts_to_sequences(X)
vocab_size = 10000+1

In [8]:
max_sequence_length = 100
X_padded = pad_sequences(X_sequences, padding='post', maxlen=max_sequence_length)
print(X_padded.shape)
print(X_padded[0])

(1375738, 100)
[6032    5   23   50  150   87   15   53 5389 1932  532 9943    3    8
    7  998   12  446    4  208 4989  131   19 1107    3  390    1 1114
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0]


### Make the datasets

In [9]:
X_tensor = torch.tensor(X_padded, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.float32)

dataset = torchdata.TensorDataset(X_tensor, y_tensor)
train_set, test_set = torchdata.random_split(dataset, [int(0.6*len(dataset)), len(dataset)-int(0.6*len(dataset))])
trainloader = torchdata.DataLoader(train_set, batch_size =4, shuffle=True)
testloader = torchdata.DataLoader(test_set, batch_size=4, shuffle=False)

### Model

In [10]:
class SimpleNNTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, input_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lin1 = nn.Linear(embedding_dim*input_len, 32)
        self.relu1 = nn.ReLU()
        self.lin2 = nn.Linear(32, 32)
        self.relu2 = nn.ReLU()
        self.lin3 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.embedding(x)
        x = x.view(x.size(0), -1)
        x = self.lin1(x)
        x = self.relu1(x)
        x = self.lin2(x)
        x = self.relu2(x)
        x = self.lin3(x)
        x = self.sigmoid(x)
        return x.squeeze(1)
    

In [43]:
net = SimpleNNTextClassifier(vocab_size=vocab_size, embedding_dim=15, input_len=max_sequence_length)
criterion = nn.BCELoss()
optimizer = optim.Adam(net.parameters(), lr=0.01)

train_loss =[]
train_accuracy = []
iters = 10

for i in range(iters):
    net.train()
    total_loss = 0
    for batch_X, batch_y in trainloader:
        pred = net(batch_X)
        loss = criterion(pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        with torch.no_grad():
            pred_class = (pred > 0.5).float()
            accuracy = (pred_class == batch_y).float().mean()


train_loss.append(total_loss/len(trainloader))
train_accuracy.append(accuracy.item())

print(f"Epoch {i+1}/{iters}- Accuracy: {accuracy.item():.4f}")


KeyboardInterrupt: 